# **Ejemplo clasificación Supervisada: Detección de Cáncer de Mama**

Este notebook tiene como propósito entender y comparar las métricas más comunes en problemas de clasificación supervisada, como:

Accuracy (Exactitud)

Precision (Precisión)

Recall (Exhaustividad o sensibilidad)

F1-score

Usaremos un caso real: detección de cáncer de mama usando el dataset Breast Cancer Wisconsin. Compararemos cómo cambian estas métricas cuando modificamos el umbral de decisión del modelo.

**📦 1. Importar librerías necesarias**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn para modelos y métricas
from sklearn.datasets import load_breast_cancer          # DataSet de Cancer de mama
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC                              # Modelo de SVM para clasificación binaria
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
)

**📁 2. Cargar el dataset**

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Ver info básica
print(f"Número de muestras: {X.shape[0]}")
print(f"Número de características: {X.shape[1]}")
X.head()

**🔎 3. Explorar la distribución de clases**

In [ ]:
# Conteo de clases
sns.countplot(x=y.map({0: 'Maligno', 1: 'Benigno'}))
plt.title('Distribución de tumores en el dataset')
plt.xlabel('Tipo de tumor')
plt.ylabel('Cantidad')
plt.show()

**🧪 4. Separar datos en entrenamiento y prueba**

En este ejemplo se incluye el parámetro `stratify=y` en la función train_test_split(). Se utiliza para garantizar que la proporción de clases en los conjuntos de entrenamiento y prueba sea la misma que en el conjunto original.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y )


**⚙️ 5. Escalar características**

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


**🤖 6. Entrenar modelo de clasificación**

Para que en el **modelo SVC** se habilite la estimación de probabilidades de pertenencia a cada clase al crearlo, la cual se utilizara mas adelante para el manejo del umbral, se debe utilizar el parámetro `probability=True`.

In [ ]:
model = SVC(kernel='linear', C=1.0, probability=True)  # C=1.0 (Equilibrado)
model.fit(X_train_scaled, y_train)

**🔮 7. Predicción y métricas con umbral 0.5 (por defecto)**

Una vez tienes las probabilidades, se puede ajustar el umbral de decisión.
Por defecto, SVM clasifica con umbral 0.5, es decir:

* Si P(clase=1) > 0.5 → predice 1

* Si P(clase=1) ≤ 0.5 → predice 0

Pero con las probabilidades se puede modificar este valor según sus necesidades:

* Por ejemplo, usar 0.3 para detectar más positivos (mayor recall).

* O usar 0.9 para ser más estricto (mayor precisión).

In [ ]:
# Predicciones estándar
y_pred = model.predict(X_test_scaled)

# Métricas
print("Evaluación con umbral 0.5 (por defecto):")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2f}")
print(f"Precisión: {precision_score(y_test, y_pred):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred):.2f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.2f}")

# Reporte de clasificación
print("REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot()
plt.title("Matriz de Confusión - Umbral 0.5")
plt.show()



**🧼 8. Cambiar umbral de decisión**

In [ ]:
# Obtener probabilidades
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Probabilidad de la clase positiva

🎯 8.1 Evaluar con umbral alto (0.9) → modelo conservador

In [ ]:
y_pred_09 = (y_proba > 0.9).astype(int)

print("Evaluación con umbral 0.9 (modelo conservador):")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_09):.2f}")
print(f"Precisión: {precision_score(y_test, y_pred_09):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred_09):.2f}")
print(f"F1-score:  {f1_score(y_test, y_pred_09):.2f}")

cm_09 = confusion_matrix(y_test, y_pred_09)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_09, display_labels=data.target_names)
disp.plot()
plt.title("Matriz de Confusión - Umbral 0.9")
plt.show()


🔍 8.2 Evaluar con umbral bajo (0.3) → modelo sensible

In [ ]:
y_pred_03 = (y_proba > 0.3).astype(int)

print("Evaluación con umbral 0.3 (modelo sensible):")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_03):.2f}")
print(f"Precisión: {precision_score(y_test, y_pred_03):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred_03):.2f}")
print(f"F1-score:  {f1_score(y_test, y_pred_03):.2f}")

cm_03 = confusion_matrix(y_test, y_pred_03)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_03, display_labels=data.target_names)
disp.plot()
plt.title("Matriz de Confusión - Umbral 0.3")
plt.show()


**📊 9. Comparar todas las métricas en una tab**la

In [ ]:
metricas = pd.DataFrame({
    'Umbral': ['0.5 (default)', '0.9 (conservador)', '0.3 (sensible)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_09),
        accuracy_score(y_test, y_pred_03)
    ],
    'Precisión': [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_09),
        precision_score(y_test, y_pred_03)
    ],
    'Recall': [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_09),
        recall_score(y_test, y_pred_03)
    ],
    'F1-score': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_09),
        f1_score(y_test, y_pred_03)
    ]
})

metricas


**🧠 10. Conclusiones**

En el diagnóstico de cáncer de mama, preferimos priorizar el recall, incluso si eso significa aceptar más falsos positivos. Es mejor que el modelo se equivoque por exceso de precaución (decir que alguien podría tener cáncer cuando no lo tiene) que por omisión (decir que no lo tiene cuando sí).

Por tanto, el modelo más “sensible” (con umbral bajo, como 0.3) puede ser más adecuado, ya que detecta más casos de cáncer, aunque sacrifique algo de precisión.

